# Chapter 22 — Retries Are Side Effects Too

**Companion to Applied AI**

Question: What happens when a timed-out — but completed — effect is naively retried?

By the end of this notebook you will have:

- implemented a non-idempotent operation with a post-effect timeout
- shown naive retry duplicating the effect
- fixed it with idempotency key, fingerprint, and effect ledger

## What this notebook demonstrates
A simulated payment/credit effect that completes but whose acknowledgement is lost. Naive retry doubles it; an idempotency key plus effect ledger replays safely.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)

seed: 42


## 1. The non-idempotent world

In [2]:
ledger = []  # the durable effect ledger: (idempotency_key, fingerprint)
balance = {"acct": 0}
class Timeout(Exception): pass

def apply_credit(amount: int):
    balance["acct"] += amount  # NOT idempotent by itself

def attempt_with_timeout_lost(key: str, amount: int, lose_ack: bool):
    apply_credit(amount)
    ledger.append({"key": key, "amount": amount})
    if lose_ack:
        raise Timeout("effect applied, acknowledgement lost")

## 2. Naive retry: timeout means 'try again' — and pays twice

In [3]:
balance["acct"] = 0
ledger.clear()
try:
    attempt_with_timeout_lost("k-1", 100, lose_ack=True)
except Timeout as e:
    print("first attempt:", e)
attempt_with_timeout_lost("k-1", 100, lose_ack=False)  # naive retry: same key, no check
print("balance after naive retry:", balance["acct"], "(should be 100)")
assert balance["acct"] == 200, "the duplicate is the lesson"
print("DUPLICATE EFFECT: 2 ledger entries, 200 credited")

first attempt: effect applied, acknowledgement lost
balance after naive retry: 200 (should be 100)
DUPLICATE EFFECT: 2 ledger entries, 200 credited


## 3. The fix: key + fingerprint + current authority, checked before acting

In [4]:
balance["acct"] = 0
ledger.clear()
def safe_attempt(key: str, amount: int, authorized: bool, lose_ack: bool = False):
    if not authorized:
        return "DENIED: no authority, 0 effects"
    for e in ledger:
        if e["key"] == key:
            if e["amount"] != amount:
                return "CONFLICT: same key, changed instruction -> refused"
            return "REPLAY: already applied -> 1 effect total, no double-apply"
    try:
        attempt_with_timeout_lost(key, amount, lose_ack=lose_ack)
    except Timeout:
        return "TIMEOUT after effect: ledger already holds it; replay, do not re-apply"
    return "APPLIED"

print(safe_attempt("k-2", 100, True, lose_ack=True))
print(safe_attempt("k-2", 100, True))
print(safe_attempt("k-2", 999, True))
print(safe_attempt("k-3", 100, False))
assert balance["acct"] == 100
assert sum(1 for e in ledger if e["key"] == "k-2") == 1

TIMEOUT after effect: ledger already holds it; replay, do not re-apply
REPLAY: already applied -> 1 effect total, no double-apply
CONFLICT: same key, changed instruction -> refused
DENIED: no authority, 0 effects


## Interpretation
- Supports: `retry ≠ replay ≠ duplicate`; FAILED ≠ no-effect; the key+fingerprint+authority check must run before acting.
- Does NOT support: a distributed transactions claim.

## Try it yourself
1. Run two threads racing the same key (the chapter's threads case) and count effects.
2. Expire the authority between attempt and replay; show DENIED.
3. Add a `replay_refused` event type to the ledger.